# 🧠 Conversation Memory in LangChain

Modern approaches to maintaining conversation context across turns, sessions, and process restarts.

## Learning Objectives
In this notebook, you will learn:
1. **Basic memory with `RunnableWithMessageHistory`** - wire per-session chat history into an LCEL chain
2. **Multi-session isolation** - give each user/session its own independent memory store
3. **Bounding memory growth** - trim messages by token count and implement a manual sliding window
4. **Summary memory** - compress older turns into a running summary while keeping recent turns verbatim
5. **Persistent memory** - back chat history with SQLite via `SQLChatMessageHistory` so it survives process restarts

## Prerequisites
- Familiarity with LangChain LCEL (`prompt | llm | parser`)
- `OPENAI_API_KEY` set in a `.env` file at the project root
- Basic understanding of chat message types (`HumanMessage`, `AIMessage`, `SystemMessage`)

> **Note**: Converted from `08_conversation_memory.py` (part of **01 LangChain Foundations**). The patterns shown here (`RunnableWithMessageHistory`, `SQLChatMessageHistory`) are LangChain's pre-1.0 "legacy" memory APIs -- they remain valid for standalone LCEL chains, but LangChain 1.x steers new **agent**-based apps toward LangGraph checkpointers for persistence instead.

---
## 🔧 Part 1: Environment Setup

We load environment variables from `.env` and initialize the chat model (`gpt-4o-mini`) that every memory demo below shares.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports & LLM Initialization
# ============================================================================
# Load environment variables from .env and initialize the chat model shared
# by every memory demo in this notebook.

from typing import Dict

from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    trim_messages,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

load_dotenv()

llm = init_chat_model("gpt-4o-mini")

print("✅ Environment variables loaded from .env")
print("🤖 LLM initialized: gpt-4o-mini (via init_chat_model)")

---
## 💬 Part 2: Basic Conversation Memory

`demo_basic_memory` wires a single LCEL chain to `RunnableWithMessageHistory`, giving the prompt a `MessagesPlaceholder` for prior turns and an in-memory store keyed by `session_id`. This is the simplest way to make a chain "remember" earlier turns within the same session.

### Key Concepts:
- **`InMemoryChatMessageHistory`**: an in-process list of messages for one session
- **`RunnableWithMessageHistory`**: wraps a chain so history is read before, and written after, each invocation
- **`session_id`**: the key used to look up (or create) a session's history

In [ ]:
# ============================================================================
# BASIC CONVERSATION MEMORY: RunnableWithMessageHistory Demo
# ============================================================================
def demo_basic_memory():
    """Basic conversation memory with RunnableWithMessageHistory."""

    print("=" * 60)
    print("BASIC CONVERSATION MEMORY")
    print("Using RunnableWithMessageHistory (modern approach)")
    print("=" * 60)

    # llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

    # Prompt with history placeholder
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a helpful assistant. Be concise."),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ]
    )

    chain = prompt | llm | StrOutputParser()

    # Session storage
    store: Dict[str, InMemoryChatMessageHistory] = {}

    def get_session_history(session_id: str) -> BaseChatMessageHistory:
        if session_id not in store:
            store[session_id] = InMemoryChatMessageHistory()
        return store[session_id]

    # Wrap with history
    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    # Configuration for this session
    config = {"configurable": {"session_id": "user_123"}}

    # Conversation
    messages = [
        "Hi! My name is Paulo.",
        "I'm learning about LangChain.",
        "What's my name and what am I learning?",
    ]

    print("\nConversation:")
    for msg in messages:
        print(f"\nUser: {msg}")
        response = chain_with_history.invoke({"input": msg}, config=config)
        print(f"AI: {response}")

    # Show stored history
    print(f"\n--- Stored History ({len(store['user_123'].messages)} messages) ---")
    for msg in store["user_123"].messages:
        role = "Human" if isinstance(msg, HumanMessage) else "AI"
        print(f"  {role}: {msg.content[:50]}...")

---
## 👥 Part 3: Multiple Conversation Sessions

`demo_multi_sessions` shows that session isolation is just a matter of the `session_id` key: each session gets its own `InMemoryChatMessageHistory`, so User A's facts never leak into User B's conversation.

In [ ]:
# ============================================================================
# MULTIPLE SESSIONS: Independent Memory per User
# ============================================================================
def demo_multi_sessions():

    print("=" * 60)
    print("MULTIPLE CONVERSATION SESSIONS")
    print("Each user gets their own memory")
    print("=" * 60)

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a helpful assistant. Remember user details."),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ]
    )

    chain = prompt | llm | StrOutputParser()

    store: Dict[str, InMemoryChatMessageHistory] = {}

    def get_session_history(session_id: str) -> BaseChatMessageHistory:
        if session_id not in store:
            store[session_id] = InMemoryChatMessageHistory()
        return store[session_id]

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    # Simulate two users
    user_a_config = {"configurable": {"session_id": "user_a"}}
    user_b_config = {"configurable": {"session_id": "user_b"}}

    # User A conversation
    print("\n--- User A ---")
    print("User A: My favorite language is Python")
    resp = chain_with_history.invoke(
        {"input": "My favorite language is Python"}, config=user_a_config
    )
    print(f"AI: {resp}")

    # User B conversation
    print("\n--- User B ---")
    print("User B: I love JavaScript")
    resp = chain_with_history.invoke(
        {"input": "I love JavaScript"}, config=user_b_config
    )
    print(f"AI: {resp}")

    # Ask each user about their preference
    print("\n--- Asking each about their preference ---")

    print("\nUser A: What's my favorite language?")
    resp = chain_with_history.invoke(
        {"input": "What's my favorite language?"}, config=user_a_config
    )
    print(f"AI: {resp}")

    print("\nUser B: What's my favorite language?")
    resp = chain_with_history.invoke(
        {"input": "What's my favorite language?"}, config=user_b_config
    )
    print(f"AI: {resp}")

---
## ✂️ Part 4: Message Trimming

Long conversations eventually exceed the LLM's context window (and cost more per call). `trim_messages` caps history by token count while optionally always preserving the system message.

### Key Concepts:
- **`strategy="last"`**: keep the most recent messages that fit within the budget
- **`token_counter`**: an LLM (or tokenizer) used to estimate token counts per message
- **`include_system`**: pin the system message so it's never trimmed away

In [ ]:
# ============================================================================
# MESSAGE TRIMMING: Bound History by Token Count
# ============================================================================
def demo_message_trimming():
    """Trim messages to fit context window."""

    print("=" * 60)
    print("MESSAGE TRIMMING")
    print("Keep conversation within token limits")
    print("=" * 60)

    # Simulate a long conversation
    messages = [
        SystemMessage(content="You are a helpful coding assistant."),
        HumanMessage(content="What is Python?"),
        AIMessage(
            content="Python is a high-level programming language known for readability and versatility. It's used in web development, data science, AI, and automation."
        ),
        HumanMessage(content="How do I install it?"),
        AIMessage(
            content="You can install Python from python.org or use package managers like apt, brew, or chocolatey. I recommend Python 3.12+ for new projects."
        ),
        HumanMessage(content="What about pip?"),
        AIMessage(
            content="Pip is Python's package installer. It comes with Python 3.4+. Use 'pip install package_name' to install packages. Consider using virtual environments with venv or uv."
        ),
        HumanMessage(content="Can you summarize everything we discussed?"),
    ]

    print(f"\nOriginal: {len(messages)} messages")

    # Trim to last N tokens
    trimmed = trim_messages(
        messages,
        max_tokens=60,
        strategy="last",
        token_counter=llm,
        include_system=True,  # Always keep system message
        allow_partial=False,
    )

    print(f"After trimming (max 60 tokens): {len(trimmed)} messages")
    print("\nTrimmed messages:")
    for msg in trimmed:
        role = type(msg).__name__.replace("Message", "")
        print(f"  {role}: {msg.content[:60]}...")

---
## 🪟 Part 5: Windowed Memory (Keep Last K)

A simpler, cruder alternative to token-based trimming: cap history to the last *k* human/AI exchange pairs by overriding `add_messages` on a custom `InMemoryChatMessageHistory` subclass. This gives predictable, fixed memory cost -- at the price of silently dropping older context.

In [ ]:
# ============================================================================
# WINDOWED MEMORY: Fixed-Size Sliding Window
# ============================================================================
def demo_windowed_memory():
    """Implement sliding window memory manually."""

    print("=" * 60)
    print("WINDOWED MEMORY (Keep Last K)")
    print("Fixed-size conversation window")
    print("=" * 60)

    class WindowedChatHistory(InMemoryChatMessageHistory):
        """Chat history that keeps only last k message pairs."""

        k: int = 3  # Pydantic field - number of exchange pairs to keep

        def add_messages(self, messages):
            super().add_messages(messages)
            # Keep only last k pairs (2k messages: human + ai)
            if len(self.messages) > self.k * 2:
                self.messages = self.messages[-(self.k * 2) :]

    store: Dict[str, WindowedChatHistory] = {}

    def get_session_history(session_id: str) -> BaseChatMessageHistory:
        if session_id not in store:
            store[session_id] = WindowedChatHistory(k=2)
        return store[session_id]

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a helpful assistant."),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ]
    )

    chain = prompt | llm | StrOutputParser()

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    config = {"configurable": {"session_id": "windowed_test"}}

    # Simulate a conversation with more than 2 pairs
    exchanges = [
        "My name is Paulo",
        "I live in Seattle",
        "I work as an AI engineer",
        "I have 2 cats",
        "What do you remember about me?",
    ]

    print("\nConversation with k=2 window:")
    for i, msg in enumerate(exchanges, 1):
        print(f"\nUser: {msg}")
        response = chain_with_history.invoke({"input": msg}, config=config)
        print(f"AI: {response}")

        # Show window state after each exchange so students SEE it sliding
        history = store["windowed_test"].messages
        print(f"  [Window: {len(history)} msgs] ", end="")
        facts_in_memory = [
            m.content[:40] for m in history if isinstance(m, HumanMessage)
        ]
        print(f"Remembers: {facts_in_memory}")

    # Final state - show what survived and what was lost
    print("\n" + "=" * 60)
    print("RESULT: Window only kept last 2 exchanges!")
    print("Lost: name (Paulo), city (Seattle), AND job (AI engineer)")
    print("Kept: cats + the 'remember' question")
    print(
        "This is the tradeoff: fixed memory = predictable cost, but older context is lost."
    )

---
## 🧾 Part 6: Summary Memory

Rather than dropping old messages, `demo_summary_memory` compresses them into a running natural-language summary once the recent-message buffer fills up. Every fact stays accessible (in compressed form) while token cost stays bounded -- a middle ground between trimming and unbounded history.

### Key Concepts:
- **Recent buffer**: the last `MAX_RECENT` messages, kept verbatim
- **Running summary**: a 2-3 sentence LLM-generated compression of everything older
- **Two chains**: one for the actual conversation, one dedicated to summarization

In [ ]:
# ============================================================================
# SUMMARY MEMORY: Compress Old Turns, Keep Recent Ones Verbatim
# ============================================================================
def demo_summary_memory():
    """End-to-end summary memory: auto-summarize old messages, keep recent ones verbatim."""

    print("=" * 60)
    print("SUMMARY MEMORY")
    print("Summarize older messages to save tokens")
    print("=" * 60)

    # --- Setup ---
    summary_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

    # The conversation prompt: summary of old context + recent messages
    chat_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful assistant. Be concise.\n\n"
                "Summary of earlier conversation:\n{summary}",
            ),
            MessagesPlaceholder(variable_name="recent_messages"),
            ("human", "{input}"),
        ]
    )

    chat_chain = chat_prompt | chat_llm | StrOutputParser()

    # The summarization prompt: compress messages into a running summary
    summarize_prompt = ChatPromptTemplate.from_template(
        "Condense the current summary and new messages into a single updated summary "
        "(2-3 sentences). Preserve all key facts about the user.\n\n"
        "Current summary:\n{current_summary}\n\n"
        "New messages:\n{new_messages}\n\n"
        "Updated summary:"
    )

    summarize_chain = summarize_prompt | summary_llm | StrOutputParser()

    # --- State ---
    running_summary = ""  # starts empty
    recent_messages = []  # full message objects
    MAX_RECENT = 4  # keep last 4 messages (2 exchanges) before summarizing

    # --- Conversation ---
    exchanges = [
        "My name is Paulo and I'm from Seattle",
        "I work as an AI engineer building RAG systems",
        "I have 2 cats named Luna and Milo",
        "I'm building a LangChain course for Udemy",
        "What do you know about me? List everything.",
    ]

    print(f"\nConfig: keep last {MAX_RECENT} messages, summarize the rest\n")

    for user_input in exchanges:
        print(f"User: {user_input}")

        # 1. Call the LLM with summary + recent messages + new input
        response = chat_chain.invoke(
            {
                "summary": (
                    running_summary if running_summary else "No prior conversation."
                ),
                "recent_messages": recent_messages,
                "input": user_input,
            }
        )
        print(f"AI: {response}")

        # 2. Add this exchange to recent messages
        recent_messages.append(HumanMessage(content=user_input))
        recent_messages.append(AIMessage(content=response))

        # 3. If recent messages exceed limit, summarize the oldest ones
        if len(recent_messages) > MAX_RECENT:
            # Take the oldest messages that will be summarized away
            messages_to_summarize = recent_messages[:-MAX_RECENT]
            formatted = "\n".join(
                f"{'Human' if isinstance(m, HumanMessage) else 'AI'}: {m.content}"
                for m in messages_to_summarize
            )

            # Update the running summary
            running_summary = summarize_chain.invoke(
                {
                    "current_summary": (
                        running_summary if running_summary else "None yet."
                    ),
                    "new_messages": formatted,
                }
            )

            # Keep only the most recent messages
            recent_messages = recent_messages[-MAX_RECENT:]

            print(
                f"  >>> Summarized! Compressed {len(messages_to_summarize)} old messages"
            )
            print(f"  >>> Summary: {running_summary}")
            print(f"  >>> Recent buffer: {len(recent_messages)} messages")
        print()

    # --- Final state ---
    print("=" * 60)
    print("FINAL MEMORY STATE")
    print("=" * 60)
    print(f"\nRunning summary (compressed old context):\n  {running_summary}")
    print(f"\nRecent messages kept verbatim ({len(recent_messages)}):")
    for msg in recent_messages:
        role = "Human" if isinstance(msg, HumanMessage) else "AI"
        print(f"  {role}: {msg.content[:80]}")
    print("\nKey insight: ALL facts preserved (name, city, job, cats, course)")
    print("But token cost stays bounded -- old messages are compressed, not deleted!")

---
## 💾 Part 7: Persistent Memory (Exercises)

Everything above lives only in a Python dict (`store`) -- it disappears the moment the process exits. These two exercises swap `InMemoryChatMessageHistory` for `SQLChatMessageHistory`, backing conversation memory with a real SQLite database so it survives restarts.

### 7.1 🗄️ `exercise_persistent_memory`

A first pass: wire `SQLChatMessageHistory` into the same `RunnableWithMessageHistory` pattern used above -- only the `get_session_history` factory changes.

In [ ]:
# ============================================================================
# EXERCISE: Persistent Memory with SQLChatMessageHistory
# ============================================================================
# Exercise
def exercise_persistent_memory():
    """
    EXERCISE: Build a chatbot with:
    1. Persistent memory (SQLite)
    2. Automatic summarization after 10 messages
    3. User preference tracking

    Hint: Combine RunnableWithMessageHistory with SQLChatMessageHistory
    """

    print("=" * 60)
    print("EXERCISE: Persistent Memory Chatbot")
    print("=" * 60)

    from langchain_community.chat_message_histories import SQLChatMessageHistory
    import os

    # Use SQLite for persistence
    db_path = "./chat_history.db"

    def get_session_history(session_id: str) -> BaseChatMessageHistory:
        return SQLChatMessageHistory(
            session_id=session_id, connection=f"sqlite:///{db_path}"
        )

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a helpful assistant. Remember user preferences."),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    config = {"configurable": {"session_id": "persistent_user"}}

    print("\nPersistent memory chatbot:")
    print("(Messages saved to SQLite database)\n")

    # Test conversation
    test_messages = [
        "Remember that I prefer dark mode themes",
        "What theme do I prefer?",
    ]

    for msg in test_messages:
        print(f"User: {msg}")
        response = chain_with_history.invoke({"input": msg}, config=config)
        print(f"AI: {response}\n")

    print(f"Database created: {db_path}")
    print("Messages persist across restarts!")

    # Cleanup for demo
    if os.path.exists(db_path):
        os.remove(db_path)

### 7.2 🔬 `exercise_persistent_memory_proof`

A stronger proof: build the chain **twice**, simulating two separate program runs against the same SQLite file, and inspect the raw database rows in between. The second chain instance starts with zero in-memory state yet still recalls everything -- proof that persistence, not memory, is doing the work.

In [ ]:
# ============================================================================
# EXERCISE: Persistent Memory Proof (Two Separate Chain Instances)
# ============================================================================
def exercise_persistent_memory_proof():
    """
    EXERCISE: Build a chatbot with:
    1. Persistent memory (SQLite)
    2. Proof that messages survive across separate chain instances
    3. User preference tracking

    Key idea: We create the chain TWICE to simulate two separate program runs.
    The second run reads from the same SQLite DB and recalls what the first run stored.
    """

    print("=" * 60)
    print("EXERCISE: Persistent Memory Chatbot")
    print("=" * 60)

    from langchain_community.chat_message_histories import SQLChatMessageHistory
    import sqlite3
    import os

    db_path = "./chat_history.db"
    connection_string = f"sqlite:///{db_path}"
    session_id = "persistent_user"

    # Clean slate
    if os.path.exists(db_path):
        os.remove(db_path)

    # --- Helper: build a fresh chain (simulates a new program run) ---
    def build_chain():
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

        def get_session_history(sid: str) -> BaseChatMessageHistory:
            return SQLChatMessageHistory(
                session_id=sid,
                connection=connection_string,
            )

        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You are a helpful assistant. Remember user preferences and facts.",
                ),
                MessagesPlaceholder(variable_name="history"),
                ("human", "{input}"),
            ]
        )

        chain = prompt | llm | StrOutputParser()

        return RunnableWithMessageHistory(
            chain,
            get_session_history,
            input_messages_key="input",
            history_messages_key="history",
        )

    config = {"configurable": {"session_id": session_id}}

    # =====================================================
    # RUN 1 -- Store preferences (simulates first session)
    # =====================================================
    print("\n--- RUN 1: Storing preferences ---\n")

    chain_v1 = build_chain()

    run1_messages = [
        "My name is Paulo. I prefer dark mode themes and Python over JavaScript.",
        "I also like my responses concise -- no fluff.",
    ]

    for msg in run1_messages:
        print(f"User: {msg}")
        response = chain_v1.invoke({"input": msg}, config=config)
        print(f"AI:   {response}\n")

    # Throw away the chain object entirely -- no in-memory state survives
    del chain_v1

    # =====================================================
    # PROOF: Inspect the raw SQLite database
    # =====================================================
    print("--- DATABASE PROOF ---\n")
    print(f"Database file exists: {os.path.exists(db_path)}")
    print(f"Database size: {os.path.getsize(db_path)} bytes\n")

    conn = sqlite3.connect(db_path)
    cursor = conn.execute("SELECT * FROM message_store ORDER BY rowid")
    rows = cursor.fetchall()
    print(f"Total messages stored in DB: {len(rows)}\n")

    for i, row in enumerate(rows):
        print(
            f"  Row {i + 1}: session={row[0] if len(row) > 0 else 'N/A'}, "
            f"message (first 80 chars): {str(row[1])[:80] if len(row) > 1 else 'N/A'}..."
        )
    conn.close()

    # =====================================================
    # RUN 2 -- Brand new chain, same DB (simulates restart)
    # =====================================================
    print("\n--- RUN 2: Fresh chain, testing recall ---\n")

    chain_v2 = build_chain()

    recall_questions = [
        "What's my name?",
        "What theme do I prefer?",
        "What programming language do I prefer?",
        "How do I like my responses?",
    ]

    for msg in recall_questions:
        print(f"User: {msg}")
        response = chain_v2.invoke({"input": msg}, config=config)
        print(f"AI:   {response}\n")

    del chain_v2

    # =====================================================
    # FINAL: Show total messages accumulated
    # =====================================================
    print("--- FINAL DATABASE STATE ---\n")
    conn = sqlite3.connect(db_path)
    cursor = conn.execute("SELECT COUNT(*) FROM message_store")
    count = cursor.fetchone()[0]
    conn.close()

    print(f"Total messages in DB after both runs: {count}")
    print("Key insight: The second chain had ZERO in-memory history.")
    print("Everything was loaded from SQLite -- true persistence!")

    # Cleanup
    if os.path.exists(db_path):
        os.remove(db_path)

---
## ▶️ Part 8: Running the Demos

The original `__main__` guard, kept verbatim -- Jupyter sets `__name__` to `"__main__"`, so this cell executes as written. Uncomment a different line to run that demo instead.

> **Note**: `exercise_persistent_memory_proof()` requires the `langchain-community` package (for `SQLChatMessageHistory`). If it isn't installed, the cell below will raise a `ModuleNotFoundError` -- install it and re-run.

In [ ]:
# ============================================================================
# RUN: Execute a Demo
# ============================================================================
if __name__ == "__main__":
    # demo_basic_memory()
    # demo_multi_sessions()
    # demo_message_trimming()
    # demo_windowed_memory()
    # demo_summary_memory()
    # exercise_persistent_memory()
    exercise_persistent_memory_proof()

---
## 📝 Summary

In this notebook, we learned:

### 1. Wiring Memory into LCEL Chains
- **`RunnableWithMessageHistory`**: wraps `prompt | llm | parser` with automatic history read/write
- **`session_id`**: the only thing that separates one conversation from another -- `demo_multi_sessions` proved isolation is free once you key your store correctly
- **`InMemoryChatMessageHistory`**: the default, process-local backing store

### 2. Bounding Memory Growth
- **Trimming** (`trim_messages`): drop old messages once a token budget is exceeded, optionally always keeping the system message
- **Windowing**: keep only the last *k* exchange pairs by overriding `add_messages` -- simple, but silently loses older facts
- **Summarization**: compress old turns into a running summary instead of deleting them -- keeps every fact accessible while bounding token cost

### 3. Persistence Beyond the Process
- **`SQLChatMessageHistory`**: swaps the in-memory store for a SQLite-backed one with the same `RunnableWithMessageHistory` interface
- The two-run proof (`exercise_persistent_memory_proof`) showed a brand-new chain object recalling facts it never saw in memory -- the database, not the Python process, is the source of truth

### Next Steps
- These are LangChain's legacy, pre-1.0 memory primitives; when building **agents** (not just single LCEL chains), prefer LangGraph's checkpointer-based persistence for the same guarantees
- Try swapping `SQLChatMessageHistory` for a Postgres or Redis-backed history in a real deployment
- Combine windowing/summarization with persistence: summarize old turns *before* writing them to SQLite to control database growth too